In [0]:
class Bronze_races():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimeType
        races_schema = StructType([
            StructField("raceId", IntegerType(), nullable=False),
            StructField("year", IntegerType(), nullable=False),
            StructField("round", IntegerType(), nullable=False),
            StructField("circuitId", IntegerType(), nullable=False),
            StructField("name", StringType(), nullable=False),
            StructField("date", StringType(), nullable=False),
            StructField("time", StringType(), nullable=True),
            StructField("url", StringType(), nullable=True)
        ])
        return races_schema

    def read_data(self):
        df= (spark.readStream
             .format("csv")
             .option("header", True)
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 2000)
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze races Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('RacesIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-races")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")#full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.races')
                                 
                    ) 
        print("Done")
        return sQuery   


In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_races_instance = Bronze_races("races",source)
Squery_Bronze_races =Bronze_races_instance.process()
Squery_Bronze_races.awaitTermination()
print("Successfully bronze-ingestion-races stream in running")
Squery_Bronze_races.stop()